In [1]:
from pathlib import Path

REPO_URL = 'https://github.com/itsyashk/profiling-and-reasoning.git'
CLONED_REPO_ROOT = Path('/content/hw2')

if (CLONED_REPO_ROOT / '.git').exists():
    !git -C {CLONED_REPO_ROOT} pull --ff-only
elif CLONED_REPO_ROOT.exists():
    print(f'Using existing non-git repo at {CLONED_REPO_ROOT}')
else:
    !git clone {REPO_URL} {CLONED_REPO_ROOT}


Cloning into '/content/hw2'...
remote: Enumerating objects: 130, done.
remote: Counting objects: 100% (130/130), done.
remote: Compressing objects: 100% (101/101), done.
remote: Total 130 (delta 31), reused 120 (delta 21), pack-reused 0 (from 0)
Receiving objects: 100% (130/130), 4.46 MiB | 1.43 MiB/s, done.
Resolving deltas: 100% (31/31), done.


In [2]:
%%writefile /content/hw2/prompt.txt
A conversation between User and Assistant. The User asks a question, and the Assistant solves it. The Assistant first thinks about the reasoning process in the mind and then provides the User with the answer. The reasoning process is enclosed within <think> </think> and answer is enclosed within <answer> </answer> tags, respectively, i.e., <think> reasoning process here </think> <answer> answer here </answer>.
User: {question}
Assistant: <think>


Overwriting /content/hw2/prompt.txt


In [3]:
%%writefile /content/hw2/alignment/prompts.py
from pathlib import Path


class LazyPromptTemplate:
    """String-like prompt template that reads the prompt file only when used."""

    def __init__(self, filename: str = "prompt.txt") -> None:
        self.filename = filename
        self._template: str | None = None

    def _load(self) -> str:
        if self._template is None:
            self._template = load_prompt_template(self.filename)
        return self._template

    def format(self, *args, **kwargs) -> str:
        return self._load().format(*args, **kwargs)

    def __str__(self) -> str:
        return self._load()

    def __repr__(self) -> str:
        return repr(self._load())

    def __eq__(self, other) -> bool:
        return self._load() == other


def load_prompt_template(filename: str = "prompt.txt") -> str:
    """Load the CoT prompt template from the repository root."""
    prompt_path = Path(__file__).resolve().parent.parent / filename
    return prompt_path.read_text(encoding="utf-8").rstrip("\n")


DIRECT_PROMPT_TEMPLATE = """Please answer with ONLY the answer enclosed in <answer> </answer> tags.
Question: {question}
"""

COT_PROMPT_TEMPLATE = LazyPromptTemplate()


Overwriting /content/hw2/alignment/prompts.py


In [4]:
%%writefile /content/hw2/alignment/rewards.py
from __future__ import annotations

from collections import Counter
from typing import Iterable

from .drgrpo_grader import grade


def extract_answer_from_tags(response: str) -> str | None:
    """Extract the final answer from a model response containing <answer> tags."""
    if "<answer>" not in response or "</answer>" not in response:
        return None
    answer = response.rsplit("<answer>", maxsplit=1)[-1].split("</answer>", maxsplit=1)[0].strip()
    return answer or None


def answer_tag_reward_fn(response: str, ground_truth: str | float | int | list[str], fast: bool = True) -> dict[str, float]:
    """Score a response that is expected to end with a tagged final answer."""
    model_answer = extract_answer_from_tags(response)
    if model_answer is None:
        return {"format_reward": 0.0, "answer_reward": 0.0, "reward": 0.0}

    if isinstance(ground_truth, (float, int)):
        ground_truth = str(ground_truth)

    if isinstance(ground_truth, str):
        is_correct = grade(model_answer, ground_truth, fast)
    else:
        is_correct = any(grade(model_answer, candidate, fast) for candidate in ground_truth)

    return {
        "format_reward": 1.0,
        "answer_reward": 1.0 if is_correct else 0.0,
        "reward": 1.0 if is_correct else 0.0,
    }


def majority_vote_tagged_answers(responses: Iterable[str]) -> str | None:
    """Return the most common tagged answer across a self-consistency sample."""
    answers = [answer for answer in (extract_answer_from_tags(response) for response in responses) if answer is not None]
    if not answers:
        return None
    return Counter(answers).most_common(1)[0][0]


Overwriting /content/hw2/alignment/rewards.py


In [5]:
%%writefile /content/hw2/alignment/eval.py
from __future__ import annotations

import json
from collections.abc import Callable, Sequence
from pathlib import Path
from typing import Any

from .prompts import COT_PROMPT_TEMPLATE, DIRECT_PROMPT_TEMPLATE
from .rewards import answer_tag_reward_fn, majority_vote_tagged_answers


DEFAULT_MODEL_NAME = "Qwen/Qwen2.5-Math-1.5B"
DEFAULT_VALIDATION_SIZE = 256


def load_gsm8k_examples(split: str) -> list[dict[str, Any]]:
    """Load GSM8K examples from HuggingFace datasets."""
    from datasets import load_dataset
    return list(load_dataset("openai/gsm8k", "main", split=split))


def build_prompts(examples: Sequence[dict[str, Any]], prompt_template: str) -> list[str]:
    """Format raw GSM8K examples into prompt strings."""
    return [str(prompt_template).format(question=ex["question"]) for ex in examples]


def _extract_ground_truth(examples: Sequence[dict[str, Any]]) -> list[str]:
    truths = []
    for ex in examples:
        answer = ex["answer"]
        if "####" in answer:
            answer = answer.split("####")[-1].strip()
        truths.append(answer)
    return truths


# ---------------------------------------------------------------------------
# vLLM path (fast batched inference, Linux/Colab only)
# ---------------------------------------------------------------------------

def evaluate_vllm(
    vllm_model,
    reward_fn: Callable[[str, str], dict[str, float]],
    prompts: Sequence[str],
    eval_sampling_params,
    ground_truths: Sequence[str] | None = None,
    num_return_sequences: int = 1,
) -> dict[str, Any]:
    """Generate with vLLM, score responses, return evaluation artifacts."""
    outputs = vllm_model.generate(list(prompts), eval_sampling_params)

    results = []
    total_format = 0.0
    total_answer = 0.0

    for i, (prompt, output) in enumerate(zip(prompts, outputs)):
        gt = ground_truths[i] if ground_truths is not None else ""

        if num_return_sequences == 1:
            response = output.outputs[0].text
            reward_info = reward_fn(response, gt)
            results.append({"prompt": prompt, "response": response, "ground_truth": gt, **reward_info})
            total_format += reward_info["format_reward"]
            total_answer += reward_info["answer_reward"]
        else:
            responses = [o.text for o in output.outputs]
            voted = majority_vote_tagged_answers(responses)
            voted_response = f"<answer> {voted} </answer>" if voted else ""
            reward_info = reward_fn(voted_response, gt)
            results.append({
                "prompt": prompt,
                "responses": responses,
                "voted_answer": voted,
                "ground_truth": gt,
                **reward_info,
            })
            total_format += reward_info["format_reward"]
            total_answer += reward_info["answer_reward"]

    n = len(prompts)
    return {
        "results": results,
        "mean_format_reward": total_format / n,
        "mean_answer_reward": total_answer / n,
        "n": n,
    }


def _load_vllm_model(model_name: str = DEFAULT_MODEL_NAME):
    from vllm import LLM
    return LLM(model=model_name, dtype="bfloat16")


def _make_sampling_params(max_tokens: int, temperature: float = 1.0, n: int = 1):
    from vllm import SamplingParams
    return SamplingParams(
        n=n,
        temperature=temperature,
        max_tokens=max_tokens,
        stop=["</answer>"],
        include_stop_str_in_output=True,
    )


# ---------------------------------------------------------------------------
# Transformers fallback path (slower, CPU/any GPU)
# ---------------------------------------------------------------------------

def _generate_with_transformers(
    model,
    tokenizer,
    prompts: list[str],
    max_new_tokens: int = 1024,
    temperature: float = 1.0,
    num_return_sequences: int = 1,
    device: str = "cuda",
    batch_size: int = 16,
) -> list[list[str]]:
    import torch
    stop = "</answer>"
    model.eval()
    all_responses: list[list[str]] = []
    do_sample = num_return_sequences > 1 or temperature != 1.0
    old_padding_side = tokenizer.padding_side
    tokenizer.padding_side = "left"

    try:
        with torch.no_grad():
            for start in range(0, len(prompts), batch_size):
                batch_prompts = prompts[start : start + batch_size]
                inputs = tokenizer(batch_prompts, return_tensors="pt", padding=True).to(device)
                input_len = inputs["input_ids"].shape[1]
                outputs = model.generate(
                    **inputs,
                    max_new_tokens=max_new_tokens,
                    do_sample=do_sample,
                    temperature=temperature if do_sample else None,
                    num_return_sequences=num_return_sequences,
                    pad_token_id=tokenizer.pad_token_id,
                    eos_token_id=tokenizer.eos_token_id,
                    use_cache=True,
                )
                decoded = []
                for seq in outputs[:, input_len:]:
                    text = tokenizer.decode(seq, skip_special_tokens=False)
                    if stop in text:
                        text = text[: text.index(stop) + len(stop)]
                    decoded.append(text)
                for i in range(0, len(decoded), num_return_sequences):
                    all_responses.append(decoded[i : i + num_return_sequences])
    finally:
        tokenizer.padding_side = old_padding_side

    return all_responses


def evaluate_transformers(
    model,
    tokenizer,
    reward_fn: Callable[[str, str], dict[str, float]],
    prompts: list[str],
    ground_truths: list[str],
    max_new_tokens: int = 1024,
    num_return_sequences: int = 1,
    temperature: float = 1.0,
    device: str = "cuda",
) -> dict[str, Any]:
    all_responses = _generate_with_transformers(
        model, tokenizer, prompts,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        num_return_sequences=num_return_sequences,
        device=device,
    )
    results = []
    total_format = total_answer = 0.0

    for prompt, responses, gt in zip(prompts, all_responses, ground_truths):
        if num_return_sequences == 1:
            response = responses[0]
            reward_info = reward_fn(response, gt)
            results.append({"prompt": prompt, "response": response, "ground_truth": gt, **reward_info})
        else:
            voted = majority_vote_tagged_answers(responses)
            voted_response = f"<answer> {voted} </answer>" if voted else ""
            reward_info = reward_fn(voted_response, gt)
            results.append({
                "prompt": prompt, "responses": responses,
                "voted_answer": voted, "ground_truth": gt, **reward_info,
            })
        total_format += reward_info["format_reward"]
        total_answer += reward_info["answer_reward"]

    n = len(prompts)
    return {
        "results": results,
        "mean_format_reward": total_format / n,
        "mean_answer_reward": total_answer / n,
        "n": n,
    }


# ---------------------------------------------------------------------------
# Utilities
# ---------------------------------------------------------------------------

def write_evaluation_results(results: dict[str, Any], output_path: Path) -> None:
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(results, f, indent=2, ensure_ascii=False)
    print(f"Saved → {output_path}")


def _load_model_and_tokenizer(model_name: str = DEFAULT_MODEL_NAME):
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token_id = tokenizer.eos_token_id
    model = AutoModelForCausalLM.from_pretrained(
        model_name, torch_dtype=torch.bfloat16, device_map="auto",
    )
    model.eval()
    return model, tokenizer


# ---------------------------------------------------------------------------
# Named runners (used by run_*.py scripts or notebook)
# ---------------------------------------------------------------------------

def run_direct_baseline(output_path: Path) -> None:
    examples = load_gsm8k_examples("test")
    prompts = build_prompts(examples, DIRECT_PROMPT_TEMPLATE)
    ground_truths = _extract_ground_truth(examples)
    model, tokenizer = _load_model_and_tokenizer()
    device = next(model.parameters()).device.type
    results = evaluate_transformers(
        model, tokenizer, answer_tag_reward_fn, prompts, ground_truths,
        max_new_tokens=512, device=device,
    )
    print(f"format: {results['mean_format_reward']:.3f}  answer: {results['mean_answer_reward']:.3f}")
    write_evaluation_results(results, output_path)


def run_cot_baseline(output_path: Path) -> None:
    examples = load_gsm8k_examples("test")
    prompts = build_prompts(examples, COT_PROMPT_TEMPLATE)
    ground_truths = _extract_ground_truth(examples)
    model, tokenizer = _load_model_and_tokenizer()
    device = next(model.parameters()).device.type
    results = evaluate_transformers(
        model, tokenizer, answer_tag_reward_fn, prompts, ground_truths,
        max_new_tokens=1024, device=device,
    )
    print(f"format: {results['mean_format_reward']:.3f}  answer: {results['mean_answer_reward']:.3f}")
    write_evaluation_results(results, output_path)


def run_self_consistency_baseline(output_path: Path, k: int = 5) -> None:
    examples = load_gsm8k_examples("test")
    prompts = build_prompts(examples, COT_PROMPT_TEMPLATE)
    ground_truths = _extract_ground_truth(examples)
    model, tokenizer = _load_model_and_tokenizer()
    device = next(model.parameters()).device.type
    results = evaluate_transformers(
        model, tokenizer, answer_tag_reward_fn, prompts, ground_truths,
        max_new_tokens=1024, num_return_sequences=k, temperature=1.0, device=device,
    )
    results["k"] = k
    print(f"format: {results['mean_format_reward']:.3f}  answer: {results['mean_answer_reward']:.3f}")
    write_evaluation_results(results, output_path)


def get_prompt_template(use_cot: bool) -> str:
    return COT_PROMPT_TEMPLATE if use_cot else DIRECT_PROMPT_TEMPLATE


Overwriting /content/hw2/alignment/eval.py


In [6]:
%%writefile /content/hw2/alignment/grpo.py
from __future__ import annotations

from collections.abc import Callable, Sequence
from typing import Any

import torch
from torch import Tensor


def tokenize_prompt_and_output(
    prompt_strs: list[str],
    output_strs: list[str],
    tokenizer,
) -> dict[str, Tensor]:
    """Tokenize prompt/output pairs and build a response mask over the labels."""
    pairs = []
    for prompt, output in zip(prompt_strs, output_strs, strict=True):
        p_ids = tokenizer.encode(prompt, add_special_tokens=False)
        o_ids = tokenizer.encode(output, add_special_tokens=False)
        pairs.append((p_ids, o_ids))

    max_len = max(len(p) + len(o) - 1 for p, o in pairs)

    input_ids_list, labels_list, mask_list = [], [], []
    for p_ids, o_ids in pairs:
        full = p_ids + o_ids
        pad_len = max_len - (len(full) - 1)
        pad = [tokenizer.pad_token_id] * pad_len
        input_ids_list.append(full[:-1] + pad)
        labels_list.append(full[1:] + pad)
        # response_mask is True where the label is a response token
        mask_list.append([False] * (len(p_ids) - 1) + [True] * len(o_ids) + [False] * pad_len)

    return {
        "input_ids": torch.tensor(input_ids_list, dtype=torch.long),
        "labels": torch.tensor(labels_list, dtype=torch.long),
        "response_mask": torch.tensor(mask_list, dtype=torch.bool),
    }


def compute_entropy(logits: Tensor) -> Tensor:
    """Compute per-token entropies over the vocabulary dimension."""
    log_probs = torch.log_softmax(logits, dim=-1)
    return -(torch.exp(log_probs) * log_probs).sum(dim=-1)


def get_response_log_probs(
    model: torch.nn.Module,
    input_ids: Tensor,
    labels: Tensor,
    return_token_entropy: bool = False,
) -> dict[str, Tensor]:
    """Score conditional log-probabilities for a batch of prompt/response examples."""
    logits = model(input_ids).logits
    log_probs = torch.log_softmax(logits, dim=-1).gather(-1, labels.unsqueeze(-1)).squeeze(-1)
    result: dict[str, Tensor] = {"log_probs": log_probs}
    if return_token_entropy:
        result["token_entropy"] = compute_entropy(logits)
    return result


def masked_normalize(
    tensor: Tensor,
    mask: Tensor,
    normalize_constant: float,
    dim: int | None = None,
) -> Tensor:
    """Sum over masked elements and normalize by the provided constant."""
    return (tensor * mask).sum(dim=dim) / normalize_constant


def compute_group_normalized_rewards(
    reward_fn: Callable[[str, str], dict[str, float]],
    rollout_responses: list[str],
    repeated_ground_truths: list[str],
    group_size: int,
    advantage_eps: float,
    normalize_by_std: bool,
) -> tuple[Tensor, Tensor, dict[str, float]]:
    """Compute raw rewards and per-group normalized advantages for GRPO."""
    raw_scores = [
        reward_fn(resp, gt)["reward"]
        for resp, gt in zip(rollout_responses, repeated_ground_truths, strict=True)
    ]
    raw_rewards = torch.tensor(raw_scores, dtype=torch.float32)

    grouped = raw_rewards.view(-1, group_size)
    centered = grouped - grouped.mean(dim=1, keepdim=True)
    if normalize_by_std:
        advantages = centered / (grouped.std(dim=1, keepdim=True, unbiased=False) + advantage_eps)
    else:
        advantages = centered

    metadata = {
        "mean_reward": raw_rewards.mean().item(),
        "std_reward": raw_rewards.std().item(),
        "max_reward": raw_rewards.max().item(),
        "min_reward": raw_rewards.min().item(),
    }
    return advantages.reshape(-1), raw_rewards, metadata


def compute_grpo_clip_loss(
    advantages: Tensor,
    policy_log_probs: Tensor,
    old_log_probs: Tensor,
    cliprange: float,
) -> tuple[Tensor, dict[str, Tensor]]:
    """Compute the per-token GRPO-Clip loss."""
    ratios = torch.exp(policy_log_probs - old_log_probs)
    clipped_ratios = torch.clamp(ratios, 1 - cliprange, 1 + cliprange)
    adv = advantages.expand_as(policy_log_probs)
    loss = -torch.minimum(ratios * adv, clipped_ratios * adv)
    metadata = {"clip_fraction": (ratios != clipped_ratios).float().mean()}
    return loss, metadata


def grpo_microbatch_train_step(
    policy_log_probs: Tensor,
    response_mask: Tensor,
    gradient_accumulation_steps: int,
    advantages: Tensor,
    old_log_probs: Tensor,
    cliprange: float,
) -> tuple[Tensor, dict[str, Tensor]]:
    """Backpropagate a single GRPO microbatch loss."""
    per_token_loss, metadata = compute_grpo_clip_loss(advantages, policy_log_probs, old_log_probs, cliprange)
    mask_f = response_mask.to(per_token_loss.dtype)
    per_example = (per_token_loss * mask_f).sum(dim=1) / response_mask.sum(dim=1)
    loss = per_example.mean() / gradient_accumulation_steps
    loss.backward()
    return loss, metadata


def log_generations(
    prompts: Sequence[str],
    responses: Sequence[str],
    ground_truths: Sequence[str],
    reward_infos: Sequence[dict[str, float]],
    token_entropies: Sequence[float] | None = None,
) -> list[dict[str, Any]]:
    """Create serializable generation logs for debugging training runs."""
    logs = []
    for i, (prompt, response, gt, reward_info) in enumerate(
        zip(prompts, responses, ground_truths, reward_infos, strict=True)
    ):
        entry: dict[str, Any] = {
            "prompt": prompt,
            "response": response,
            "ground_truth": gt,
            "reward_info": reward_info,
        }
        if token_entropies is not None:
            entry["avg_token_entropy"] = token_entropies[i]
        logs.append(entry)
    return logs


def train_grpo(
    policy_model,
    tokenizer,
    train_examples: list[dict],
    val_examples: list[dict],
    reward_fn: Callable,
    prompt_template: str,
    n_grpo_steps: int = 8,
    learning_rate: float = 1e-5,
    advantage_eps: float = 1e-6,
    rollout_batch_size: int = 32,
    group_size: int = 8,
    sampling_temperature: float = 1.0,
    sampling_min_tokens: int = 4,
    sampling_max_tokens: int = 256,
    epochs_per_rollout_batch: int = 1,
    train_batch_size: int = 32,
    gradient_accumulation_steps: int = 16,
    cliprange: float = 1.0,
    normalize_by_std: bool = True,
    log_every: int = 1,
    val_every: int = 5,
    val_size: int = 256,
    device: str = "cuda",
    wandb_run=None,
) -> dict[str, Any]:
    """Run the full GRPO training loop from Section 3.5."""
    import random
    from transformers import GenerationConfig

    policy_model.train()
    optimizer = torch.optim.Adam(
        policy_model.parameters(),
        lr=learning_rate,
        weight_decay=0.0,
        betas=(0.9, 0.95),
    )

    n_prompts_per_rollout = rollout_batch_size // group_size
    micro_batch_size = train_batch_size // gradient_accumulation_steps
    n_microbatches = rollout_batch_size // micro_batch_size

    from .eval import _extract_ground_truth, build_prompts

    val_prompts = build_prompts(val_examples[:val_size], prompt_template)
    val_ground_truths = _extract_ground_truth(val_examples[:val_size])

    history: dict[str, list] = {
        "step": [],
        "val_reward": [],
        "train_reward": [],
        "val_generations": [],
    }

    stop_strings = ["</answer>"]

    gen_config = GenerationConfig(
        do_sample=True,
        temperature=sampling_temperature,
        max_new_tokens=sampling_max_tokens,
        min_new_tokens=sampling_min_tokens,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

    def generate_responses(prompts: list[str], batch_size: int = 16) -> list[str]:
        policy_model.eval()
        responses = []
        old_padding_side = tokenizer.padding_side
        tokenizer.padding_side = "left"
        try:
            with torch.no_grad():
                for start in range(0, len(prompts), batch_size):
                    batch_prompts = prompts[start : start + batch_size]
                    inputs = tokenizer(batch_prompts, return_tensors="pt", padding=True).to(device)
                    prompt_len = inputs["input_ids"].shape[1]
                    output_ids = policy_model.generate(**inputs, generation_config=gen_config)
                    for seq in output_ids[:, prompt_len:]:
                        text = tokenizer.decode(seq, skip_special_tokens=False)
                        for stop in stop_strings:
                            if stop in text:
                                text = text[: text.index(stop) + len(stop)]
                        responses.append(text)
        finally:
            tokenizer.padding_side = old_padding_side
            policy_model.train()
        return responses

    def evaluate(prompts: list[str], ground_truths: list[str]) -> tuple[float, list[dict[str, Any]]]:
        responses = generate_responses(prompts)
        reward_infos = [reward_fn(r, gt) for r, gt in zip(responses, ground_truths)]
        rewards = [info["reward"] for info in reward_infos]
        return sum(rewards) / len(rewards), log_generations(
            prompts[:3],
            responses[:3],
            ground_truths[:3],
            reward_infos[:3],
        )

    for step in range(1, n_grpo_steps + 1):
        # Sample a batch of questions
        batch_examples = random.sample(train_examples, n_prompts_per_rollout)
        batch_prompts = build_prompts(batch_examples, prompt_template)
        batch_ground_truths = _extract_ground_truth(batch_examples)

        # Repeat each prompt group_size times
        repeated_prompts = [p for p in batch_prompts for _ in range(group_size)]
        repeated_gts = [gt for gt in batch_ground_truths for _ in range(group_size)]

        # Generate rollouts
        rollout_responses = generate_responses(repeated_prompts)

        # Compute advantages
        advantages, raw_rewards, reward_metadata = compute_group_normalized_rewards(
            reward_fn=reward_fn,
            rollout_responses=rollout_responses,
            repeated_ground_truths=repeated_gts,
            group_size=group_size,
            advantage_eps=advantage_eps,
            normalize_by_std=normalize_by_std,
        )

        # Tokenize all rollouts
        tokenized = tokenize_prompt_and_output(repeated_prompts, rollout_responses, tokenizer)
        input_ids = tokenized["input_ids"].to(device)
        labels = tokenized["labels"].to(device)
        response_mask = tokenized["response_mask"].to(device)
        advantages_dev = advantages.to(device)

        # Compute old log probs in microbatches to avoid OOM on large vocab/long seqs
        old_log_probs_chunks = []
        with torch.no_grad():
            for i in range(0, len(input_ids), micro_batch_size):
                chunk_out = get_response_log_probs(
                    policy_model,
                    input_ids[i : i + micro_batch_size],
                    labels[i : i + micro_batch_size],
                )
                old_log_probs_chunks.append(chunk_out["log_probs"].detach())
        old_log_probs = torch.cat(old_log_probs_chunks, dim=0)

        # Training epochs on this rollout batch
        for _epoch in range(epochs_per_rollout_batch):
            optimizer.zero_grad()
            total_loss = 0.0
            for mb in range(n_microbatches):
                start = mb * micro_batch_size
                end = start + micro_batch_size
                mb_input_ids = input_ids[start:end]
                mb_labels = labels[start:end]
                mb_mask = response_mask[start:end]
                mb_adv = advantages_dev[start:end].unsqueeze(1)
                mb_old = old_log_probs[start:end]

                out = get_response_log_probs(policy_model, mb_input_ids, mb_labels)
                mb_log_probs = out["log_probs"]

                loss, _ = grpo_microbatch_train_step(
                    policy_log_probs=mb_log_probs,
                    response_mask=mb_mask,
                    gradient_accumulation_steps=gradient_accumulation_steps,
                    advantages=mb_adv,
                    old_log_probs=mb_old,
                    cliprange=cliprange,
                )
                total_loss += loss.item()

            grad_norm = torch.nn.utils.clip_grad_norm_(policy_model.parameters(), 1.0)
            optimizer.step()

        train_reward = raw_rewards.mean().item()
        history["step"].append(step)
        history["train_reward"].append(train_reward)

        if step % log_every == 0:
            print(f"Step {step}: train_reward={train_reward:.3f} loss={total_loss:.4f} grad_norm={grad_norm:.3f}")

        if step % val_every == 0:
            val_reward, val_generations = evaluate(val_prompts, val_ground_truths)
            history["val_reward"].append(val_reward)
            history["val_generations"].append({"step": step, "examples": val_generations})
            print(f"Step {step}: val_reward={val_reward:.3f}")
            if wandb_run is not None:
                wandb_run.log({"val_reward": val_reward, "train_reward": train_reward, "step": step})

    return history


Overwriting /content/hw2/alignment/grpo.py


# EE/CS 148B HW2: Colab Setup

This notebook is meant to be imported directly into Colab to provide a quickstart environment setup.


## Colab Setup

Before running:

- Switch the runtime to a GPU runtime.
- We recommend using an A100 runtime for the GRPO experiments.
- The first cell clones or updates the repo at `/content/hw2`.

This notebook is set up so `Runtime -> Run all` works from a fresh GPU runtime using Transformers only. vLLM and flash-attn are optional experiments, not required for the assignment path.

Note: We will be using `pip` for dependencies inside Colab.


In [7]:
%%capture
!pip -q install "torch==2.8.0" "torchaudio==2.8.0" "torchvision==0.23.0" "transformers==4.56.1" "datasets>=3.0.0,<4" "accelerate>=1.0,<2" sentencepiece matplotlib pandas tqdm sympy "pylatexenc==2.10" latex2sympy2_extended "math-verify[antlr4-13-2]>=0.7.0" pytest


The main path intentionally avoids `vllm` and `flash-attn`, matching the updated staff recommendation. Set `USE_VLLM = True` only if you want to separately experiment with vLLM speedups for the Section 3.1/3.2 baselines, and install it in a fresh runtime after the core packages.


In [8]:
# OPTIONAL_VLLM_INSTALL
# Run this only in a fresh runtime if you want to experiment with vLLM for baselines.
# Keep USE_VLLM = False for the staff-recommended Transformers-only assignment path.
# !pip -q install "vllm==0.10.2"


In [9]:
from pathlib import Path

REPO_ROOT = CLONED_REPO_ROOT
assert REPO_ROOT.exists(), f'Repo root does not exist: {REPO_ROOT}'
print('Using repo:', REPO_ROOT)


Using repo: /content/hw2


In [10]:
import os
import sys

sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT / 'basics'))
os.chdir(REPO_ROOT)
print('cwd =', os.getcwd())

cwd = /content/hw2


In [11]:
import gc
import json
import random
import subprocess
from collections import Counter
from pathlib import Path
from types import SimpleNamespace

import matplotlib.pyplot as plt
import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed

from alignment.eval import load_gsm8k_examples, build_prompts, evaluate_vllm
from alignment.prompts import DIRECT_PROMPT_TEMPLATE, COT_PROMPT_TEMPLATE
from alignment.rewards import answer_tag_reward_fn
from alignment.grpo import train_grpo

SEED = 0
set_seed(SEED)
random.seed(SEED)

if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.set_float32_matmul_precision('high')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device =', DEVICE)
if torch.cuda.is_available():
    print('gpu =', torch.cuda.get_device_name(0))
    try:
        print(subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True, check=False).stdout)
    except FileNotFoundError:
        pass


/content/hw2/alignment/drgrpo_grader.py:45: SyntaxWarning: invalid escape sequence '\{'
  m = re.search("^\\\\text\{(?P<text>.+?)\}$", answer)
/content/hw2/alignment/drgrpo_grader.py:320: SyntaxWarning: invalid escape sequence '\%'
  string = string.replace("\%", "")
/content/hw2/alignment/drgrpo_grader.py:673: SyntaxWarning: invalid escape sequence '\^'
  BAD_REGEXES = ["\^[0-9]+\^", "\^[0-9][0-9]+"]
/content/hw2/alignment/drgrpo_grader.py:673: SyntaxWarning: invalid escape sequence '\^'
  BAD_REGEXES = ["\^[0-9]+\^", "\^[0-9][0-9]+"]
/content/hw2/alignment/drgrpo_grader.py:753: SyntaxWarning: invalid escape sequence '\d'
  p1 = re.compile("(\d)(,)(\d\d\d)($|\D)")
/content/hw2/alignment/drgrpo_grader.py:768: SyntaxWarning: invalid escape sequence '\{'
  m = re.search("^\\\\text\{(?P<text>.+?)\}$", expr)
/content/hw2/alignment/drgrpo_grader.py:801: SyntaxWarning: invalid escape sequence '\^'
  expr = re.sub(f"{unit}(es)?(s)? *(\^[0-9]+)?", "", expr)
/content/hw2/alignment/drgrpo_grader

device = cuda
gpu = NVIDIA A100-SXM4-40GB
GPU 0: NVIDIA A100-SXM4-40GB (UUID: GPU-8df82cd5-ea9b-412d-79b9-54317975dd02)



# Your code starts here!

## Config

In [12]:
MODEL_NAME = 'Qwen/Qwen2.5-Math-1.5B'
USE_VLLM   = False  # staff-recommended path: Transformers only; set True only for optional vLLM experiments
EVAL_N     = None   # None = full GSM8K test split; set 256 only for smoke tests
GRPO_STEPS = 50
K_SC       = 5      # self-consistency K

ARTIFACTS = REPO_ROOT / 'artifacts' / 'colab_results'
ARTIFACTS.mkdir(parents=True, exist_ok=True)

from alignment.eval import (
    load_gsm8k_examples, build_prompts, _extract_ground_truth,
    evaluate_vllm, evaluate_transformers, write_evaluation_results,
    _load_vllm_model, _load_model_and_tokenizer, _make_sampling_params,
)
from alignment.rewards import answer_tag_reward_fn, extract_answer_from_tags
import time

import transformers
if USE_VLLM:
    try:
        import vllm
        print('vllm =', vllm.__version__)
    except Exception as exc:
        print('vllm import failed:', repr(exc))
        print('Continuing with Transformers. To experiment with vLLM, install vllm in a fresh runtime and rerun.')
        USE_VLLM = False
print('torch =', torch.__version__, 'cuda =', torch.version.cuda)
print('transformers =', transformers.__version__)


torch = 2.8.0+cu128 cuda = 12.8
transformers = 4.56.1


## Load dataset

In [13]:
all_test_examples = load_gsm8k_examples('test')
test_examples  = all_test_examples if EVAL_N is None else all_test_examples[:EVAL_N]
train_examples = load_gsm8k_examples('train')
val_examples   = load_gsm8k_examples('test')
ground_truths  = _extract_ground_truth(test_examples)
prompts_direct = build_prompts(test_examples, DIRECT_PROMPT_TEMPLATE)
prompts_cot    = build_prompts(test_examples, COT_PROMPT_TEMPLATE)
print(f'Eval: {len(test_examples)}  Train: {len(train_examples)}')


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

main/train-00000-of-00001.parquet:   0%|          | 0.00/2.31M [00:00<?, ?B/s]

main/test-00000-of-00001.parquet:   0%|          | 0.00/419k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

Eval: 1319  Train: 7473


## §3.1 / §3.2 — Baselines

Skips automatically if results already saved to disk — safe to re-run after restart.

In [14]:
p31  = ARTIFACTS / 'direct_baseline.json'
pcot = ARTIFACTS / 'cot_baseline.json'
psc  = ARTIFACTS / f'self_consistency_k{K_SC}.json'

def evaluate_with_transformers(prompts, max_new_tokens, num_return_sequences=1):
    model, tokenizer = _load_model_and_tokenizer(MODEL_NAME)
    device = next(model.parameters()).device.type
    out = evaluate_transformers(
        model, tokenizer, answer_tag_reward_fn, prompts, ground_truths,
        max_new_tokens=max_new_tokens, num_return_sequences=num_return_sequences,
        temperature=1.0, device=device,
    )
    del model, tokenizer; gc.collect(); torch.cuda.empty_cache()
    return out

if p31.exists() and pcot.exists() and psc.exists():
    r31    = json.load(open(p31))
    r_cot  = json.load(open(pcot))
    r_sc   = json.load(open(psc))
    print('Loaded baseline results from disk; skipping generation.')
else:
    if USE_VLLM:
        print('Running vLLM baselines ...')
        llm = _load_vllm_model(MODEL_NAME)

        t0  = time.time()
        r31 = evaluate_vllm(llm, answer_tag_reward_fn, prompts_direct,
                            _make_sampling_params(512, n=1),
                            ground_truths=ground_truths, num_return_sequences=1)
        r31['elapsed_seconds'] = time.time() - t0
        write_evaluation_results(r31, p31)

        t0    = time.time()
        r_cot = evaluate_vllm(llm, answer_tag_reward_fn, prompts_cot,
                              _make_sampling_params(1024, n=1),
                              ground_truths=ground_truths, num_return_sequences=1)
        r_cot['elapsed_seconds'] = time.time() - t0
        write_evaluation_results(r_cot, pcot)

        t0   = time.time()
        r_sc = evaluate_vllm(llm, answer_tag_reward_fn, prompts_cot,
                             _make_sampling_params(1024, n=K_SC),
                             ground_truths=ground_truths, num_return_sequences=K_SC)
        r_sc['k'] = K_SC
        r_sc['elapsed_seconds'] = time.time() - t0
        write_evaluation_results(r_sc, psc)

        del llm; gc.collect(); torch.cuda.empty_cache()
        print('\n*** RESTART SESSION NOW before running GRPO (Runtime -> Restart session) ***')
        print('Then re-run from the top; baselines will be skipped automatically.')
    else:
        print('Running slower Transformers baselines ...')
        t0 = time.time(); r31 = evaluate_with_transformers(prompts_direct, 512, 1); r31['elapsed_seconds'] = time.time() - t0; write_evaluation_results(r31, p31)
        t0 = time.time(); r_cot = evaluate_with_transformers(prompts_cot, 1024, 1); r_cot['elapsed_seconds'] = time.time() - t0; write_evaluation_results(r_cot, pcot)
        t0 = time.time(); r_sc = evaluate_with_transformers(prompts_cot, 1024, K_SC); r_sc['k'] = K_SC; r_sc['elapsed_seconds'] = time.time() - t0; write_evaluation_results(r_sc, psc)

for name, r in [('direct', r31), ('CoT', r_cot), (f'SC K={K_SC}', r_sc)]:
    print(f"  {name:10s}: format={r['mean_format_reward']:.3f} answer={r['mean_answer_reward']:.3f} n={r['n']}")


Running slower Transformers baselines ...


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

Saved → /content/hw2/artifacts/colab_results/direct_baseline.json
Saved → /content/hw2/artifacts/colab_results/cot_baseline.json


KeyboardInterrupt: 

In [ ]:
def category_key(item):
    fmt = int(item.get('format_reward', 0))
    ans = int(item.get('answer_reward', 0))
    return f'format_{fmt}_answer_{ans}'

def summarize_single(results, max_examples=10):
    counts = Counter(category_key(item) for item in results['results'])
    examples = {key: [] for key in counts}
    for item in results['results']:
        key = category_key(item)
        if len(examples[key]) < max_examples:
            examples[key].append(item)
    return {'counts': dict(counts), 'examples': examples}

def summarize_self_consistency(results):
    tie_count = 0
    vote_summaries = []
    for item in results['results']:
        answers = [a for a in (extract_answer_from_tags(r) for r in item['responses']) if a]
        votes = Counter(answers)
        top = votes.most_common()
        is_tie = len(top) > 1 and top[0][1] == top[1][1]
        tie_count += int(is_tie)
        vote_summaries.append({
            'ground_truth': item['ground_truth'],
            'voted_answer': item.get('voted_answer'),
            'votes': dict(votes),
            'tie': is_tie,
            'answer_reward': item['answer_reward'],
        })
    return {
        'tie_count': tie_count,
        'tie_rate': tie_count / max(1, len(results['results'])),
        'vote_summaries': vote_summaries[:50],
    }

baseline_summary = {
    'direct': summarize_single(r31),
    'cot': summarize_single(r_cot),
    'self_consistency': summarize_self_consistency(r_sc),
    'metrics': {
        'direct': {'format': r31['mean_format_reward'], 'answer': r31['mean_answer_reward'], 'n': r31['n']},
        'cot': {'format': r_cot['mean_format_reward'], 'answer': r_cot['mean_answer_reward'], 'n': r_cot['n']},
        'self_consistency': {'format': r_sc['mean_format_reward'], 'answer': r_sc['mean_answer_reward'], 'n': r_sc['n'], 'k': K_SC},
    },
}
json.dump(baseline_summary, open(ARTIFACTS / 'baseline_summary.json', 'w'), indent=2)
print(json.dumps(baseline_summary['metrics'], indent=2))
print('Saved', ARTIFACTS / 'baseline_summary.json')


## §3.5 — GRPO training

**Only run these cells after a fresh session restart** (no vLLM in memory).

In [ ]:
import importlib
import alignment.grpo as _gm
importlib.reload(_gm)
from alignment.grpo import train_grpo
print('train_grpo reloaded from repository source.')


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

tokenizer_g = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer_g.pad_token_id is None:
    tokenizer_g.pad_token_id = tokenizer_g.eos_token_id

GRPO_KWARGS = dict(
    tokenizer=tokenizer_g,
    train_examples=train_examples, val_examples=val_examples,
    reward_fn=answer_tag_reward_fn,
    prompt_template=str(COT_PROMPT_TEMPLATE),
    n_grpo_steps=GRPO_STEPS, learning_rate=1e-5, advantage_eps=1e-6,
    rollout_batch_size=32, group_size=8, sampling_temperature=1.0,
    sampling_min_tokens=4, sampling_max_tokens=256,
    epochs_per_rollout_batch=1, train_batch_size=32,
    gradient_accumulation_steps=16, cliprange=1.0,
    log_every=1, val_every=5, val_size=256, device=str(DEVICE),
)


In [ ]:
# Run 1: with std normalization
policy_std = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, dtype=torch.bfloat16, device_map='auto')
policy_std.train()
t0 = time.time()
hist_std = train_grpo(policy_model=policy_std, normalize_by_std=True, **GRPO_KWARGS)
hist_std['elapsed_seconds'] = time.time() - t0
print(f"Done in {hist_std['elapsed_seconds']/60:.1f} min")
json.dump(hist_std, open(ARTIFACTS / 'grpo_history_std.json', 'w'), indent=2)
del policy_std; gc.collect(); torch.cuda.empty_cache()


In [ ]:
# Run 2: without std normalization
policy_nostd = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, dtype=torch.bfloat16, device_map='auto')
policy_nostd.train()
t0 = time.time()
hist_nostd = train_grpo(policy_model=policy_nostd, normalize_by_std=False, **GRPO_KWARGS)
hist_nostd['elapsed_seconds'] = time.time() - t0
print(f"Done in {hist_nostd['elapsed_seconds']/60:.1f} min")
json.dump(hist_nostd, open(ARTIFACTS / 'grpo_history_nostd.json', 'w'), indent=2)
del policy_nostd; gc.collect(); torch.cuda.empty_cache()


## Plot + download

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for hist, label in [(hist_std, 'with std norm'), (hist_nostd, 'without std norm')]:
    val_steps = list(range(5, GRPO_STEPS + 1, 5))[:len(hist['val_reward'])]
    ax.plot(val_steps, hist['val_reward'], marker='o', label=label)
ax.set_xlabel('GRPO step'); ax.set_ylabel('Val answer reward')
ax.set_title('GRPO validation reward'); ax.legend(); ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(ARTIFACTS / 'grpo_training_curves.png', dpi=150)
plt.show()


In [ ]:
import zipfile
zip_path = REPO_ROOT / 'colab_results.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for p in ARTIFACTS.rglob('*'):
        if p.is_file(): zf.write(p, p.relative_to(REPO_ROOT))
print(f'Zipped: {zip_path}  ({zip_path.stat().st_size/1024**2:.1f} MB)')
from google.colab import files; files.download(str(zip_path))
